# 00 · Data Audit

Verify the panel is fully simulated and describe key dimensions before any analysis.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

DATA_DIR = Path("../data/processed")
OUT_DIR  = Path("../outputs")
OUT_DIR.mkdir(exist_ok=True)

panel  = pd.read_parquet(DATA_DIR / "ship_month_panel.parquet")
events = pd.read_csv(DATA_DIR / "category_change_events.csv", parse_dates=["event_month"])

print(f"Panel: {panel.shape}  |  Events: {len(events)}")

## 1 · Simulated flag

In [ ]:
assert panel["is_simulated"].all(), "Found non-simulated rows — abort!"
print("✓ All rows are simulated (is_simulated=True)")

## 2 · Panel dimensions

In [ ]:
n_ships  = panel["ship_id"].nunique()
n_months = panel["month"].nunique()
date_min = panel["month"].min().date()
date_max = panel["month"].max().date()

print(f"Ships  : {n_ships}")
print(f"Months : {n_months}  ({date_min} → {date_max})")
print(f"Rows   : {len(panel):,}  (balanced: {len(panel) == n_ships * n_months})")

## 3 · Treatment breakdown

In [ ]:
n_treated = events["ship_id"].nunique()
print(f"Treated ships : {n_treated} / {n_ships}  ({n_treated/n_ships*100:.1f}%)")
print(f"Refit events  : {len(events)}")
print()
print("Event delta_categories distribution:")
print(events["delta_categories"].value_counts().sort_index())
print()
print("Event timing (months from panel start):")
print(events["event_month_idx"].describe().round(1))

## 4 · Missingness

In [ ]:
missing = panel.isnull().sum()
print(missing[missing > 0] if missing.any() else "No missing values")

## 5 · Key correlation check (the whole point of this project)

In [ ]:
# Cross-sectional: confounded
xs_corr = panel["cabin_category_count"].corr(panel["log_revenue_per_berth"])

# Within-ship: what panel FE will exploit
within_corrs = []
for _, grp in panel.groupby("ship_id"):
    if grp["cabin_category_count"].std() > 0:
        within_corrs.append(
            grp["cabin_category_count"].corr(grp["log_revenue_per_berth"])
        )
within_corr = np.mean(within_corrs)

print("Correlation check (known true β = 0.06)")
print(f"  Cross-sectional corr(categories, log_rpb) = {xs_corr:.3f}  ← confounded by scale")
print(f"  Mean within-ship corr (treated ships only) = {within_corr:.3f}  ← closer to signal")
print()
print("This gap is the entire identification problem. Notebooks 03–08 address it.")

## 6 · Descriptive stats by brand tier

In [ ]:
summary = (
    panel.groupby("brand_tier")
    .agg(
        ships=("ship_id", "nunique"),
        mean_categories=("cabin_category_count", "mean"),
        mean_log_rpb=("log_revenue_per_berth", "mean"),
        mean_berth_capacity=("berth_capacity", "mean"),
    )
    .round(3)
)
print(summary)